## Cell 1 - Install everything needed

In [ ]:
!pip install transformers -q
!pip install easyocr -q
!pip install torch torchvision -q
!pip install opencv-python-headless -q
!pip install Pillow -q
!pip install matplotlib -q
!pip install gradio -q
!pip uninstall accelerate -y -q   # this one causes a bug with TrOCR, better to remove it

print("done! restart runtime and run from top")

## Cell 2 - Imports

In [ ]:
import os
import urllib.request

import cv2
import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.patches as patches

import easyocr
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
# TrOCRProcessor - converts image into tensor the model understands
# VisionEncoderDecoderModel - the actual TrOCR model (image encoder + text decoder)

import torch
import gradio as gr

print("imports ok")
print("pytorch:", torch.__version__)
print("gpu available:", torch.cuda.is_available())

## Cell 3 - Load image
Either download from a URL or upload from your PC. Pick one option.

In [ ]:
def download_image(url, save_path):
    # downloads image from internet and saves it locally
    urllib.request.urlretrieve(url, save_path)
    print("saved to:", save_path)
    return save_path


def upload_image():
    # opens a file picker so you can upload from your PC (only works in Colab)
    from google.colab import files
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
    print("uploaded:", path)
    return path


def load_image(image_path):
    # reads the image file and shows it in the notebook
    img_bgr = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)  # cv2 reads BGR by default, convert to RGB

    plt.figure(figsize=(8, 5))
    plt.imshow(img_rgb)
    plt.title("input image")
    plt.axis('off')
    plt.show()

    print("image shape:", img_rgb.shape)  # (height, width, channels)
    return img_rgb


# --- pick one option below ---

# option A: download from URL
# IMAGE_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/af/Atomist_quote_from_Democritus.png/800px-Atomist_quote_from_Democritus.png"
# IMAGE_PATH = "input_image.png"
# download_image(IMAGE_URL, IMAGE_PATH)

# option B: upload from your PC
IMAGE_PATH = upload_image()

input_image = load_image(IMAGE_PATH)

## Cell 4 - Image Analyzer (Agent Step 1)

Before picking a model, the agent needs to understand what kind of image it is dealing with.
These 4 functions each measure one property of the image.

In [ ]:
def measure_contrast(image):
    # contrast tells us how sharp the difference is between light and dark pixels
    # high contrast = clean printed text, low contrast = scene photo or faded text
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    score = float(np.std(gray))  # standard deviation of brightness = contrast
    print(f"  contrast: {score:.2f}  (above 50 = high, below 50 = low)")
    return score


def measure_noise(image):
    # noise = random pixel variations that are not actual content
    # we blur the image slightly and compare - the difference is the noise
    # high noise = camera photo, low noise = clean digital image
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)  # small blur removes noise but keeps content
    score = float(np.mean(cv2.absdiff(gray, blurred)))
    print(f"  noise: {score:.2f}  (above 8 = noisy photo, below 8 = clean)")
    return score


def measure_edge_density(image):
    # edges are boundaries between light and dark - text has lots of sharp edges
    # we count what fraction of pixels are edges
    # high edge density = document with lots of text, low = smooth scene photo
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)  # canny finds sharp pixel boundaries
    score = float(np.sum(edges > 0)) / edges.size
    print(f"  edge density: {score:.4f}  (above 0.05 = document, below = scene)")
    return score


def measure_color_variance(image):
    # documents are mostly black and white so R, G, B channels are similar
    # scene photos have lots of different colors so channels vary a lot
    r = image[:, :, 0].astype(float)
    g = image[:, :, 1].astype(float)
    b = image[:, :, 2].astype(float)
    score = float((np.std(r - g) + np.std(r - b)) / 2)
    print(f"  color variance: {score:.2f}  (above 20 = colorful scene, below = document)")
    return score


def analyze_image(image):
    # runs all 4 measurements and returns them as a dict
    # this gives the agent the information it needs to make a decision
    print("analyzing image...")
    features = {
        'contrast'     : measure_contrast(image),
        'noise'        : measure_noise(image),
        'edge_density' : measure_edge_density(image),
        'color_var'    : measure_color_variance(image)
    }
    print("analysis done")
    return features


image_features = analyze_image(input_image)

## Cell 5 - Image Classifier (Agent Step 2)

Uses the 4 measurements from above to decide what type of image this is.

In [ ]:
def classify_image(features):
    # uses the feature scores to classify image into one of 3 types:
    #   scene      - real world photo with text (street signs, banners, photos)
    #   handwritten - handwritten notes or forms
    #   document   - clean printed or digital text (pdf, screenshot, typed doc)

    contrast     = features['contrast']
    noise        = features['noise']
    edge_density = features['edge_density']
    color_var    = features['color_var']

    if color_var > 20 or noise > 8:
        # colorful or noisy = real world photo
        image_type = 'scene'
        reason = f"color_var={color_var:.1f} or noise={noise:.1f} suggests a scene photo"

    elif edge_density < 0.03 and contrast < 60:
        # soft edges and medium contrast = handwriting (pen strokes are not as sharp as print)
        image_type = 'handwritten'
        reason = f"low edges ({edge_density:.4f}) and medium contrast ({contrast:.1f}) suggests handwriting"

    else:
        # everything else = clean printed or digital document
        image_type = 'document'
        reason = f"high contrast ({contrast:.1f}) and low noise ({noise:.1f}) suggests a document"

    print(f"image type: {image_type}")
    print(f"reason: {reason}")
    return image_type


image_type = classify_image(image_features)

## Cell 6 - Model Selector (Agent Step 3)

Now that we know the image type, pick the best pretrained model for it.

In [ ]:
def select_model(image_type):
    # each image type has a model that was specifically trained for it
    # scene photos   -> EasyOCR   (trained on real world scene text)
    # documents      -> TrOCR printed (trained on printed documents)
    # handwritten    -> TrOCR handwritten (trained on handwriting)

    model_map = {
        'scene'       : 'easyocr',
        'document'    : 'trocr-printed',
        'handwritten' : 'trocr-handwritten'
    }

    model_name = model_map[image_type]
    print(f"selected model: {model_name}")
    return model_name


selected_model = select_model(image_type)

## Cell 7 - Preprocessor (Agent Step 4)

Different models need different preprocessing before they can read the image.
EasyOCR works on color images at normal size.
TrOCR needs the image upscaled because it reads in small 16x16 patches and needs enough pixels.

In [ ]:
def preprocess_for_easyocr(image):
    # EasyOCR just needs the image resized to a reasonable width
    # it was trained on color images so we keep it in color
    h, w = image.shape[:2]
    scale = min(1200 / w, 1000 / h)
    if scale < 1.0:
        new_w = int(w * scale)
        new_h = int(h * scale)
        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    print(f"  resized to: {image.shape[1]}x{image.shape[0]}")
    return image


def preprocess_for_trocr(image):
    # TrOCR splits image into 16x16 patches and reads each one
    # if the image is too small, characters are only a few pixels wide and look identical
    # upscaling 3x gives the model more pixels per character to work with
    h, w = image.shape[:2]
    image = cv2.resize(image, (w * 3, h * 3), interpolation=cv2.INTER_CUBIC)
    print(f"  upscaled to: {image.shape[1]}x{image.shape[0]}")
    return image


def preprocess_image(image, model_name):
    # routes to the right preprocessing function based on which model was selected
    print(f"preprocessing for {model_name}...")

    if model_name == 'easyocr':
        processed = preprocess_for_easyocr(image)
    elif model_name in ('trocr-printed', 'trocr-handwritten'):
        processed = preprocess_for_trocr(image)
    else:
        processed = image.copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(image)
    axes[0].set_title("original")
    axes[0].axis('off')
    axes[1].imshow(processed)
    axes[1].set_title(f"after preprocessing ({model_name})")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    return processed


processed_image = preprocess_image(input_image, selected_model)

## Cell 8 - Model Loader (Agent Step 5)

Load only the model that was selected. No point loading all 3 models into memory.

In [ ]:
def load_easyocr():
    print("loading EasyOCR...")
    # loads CRAFT (text detection) + CRNN (text recognition) pretrained weights
    reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
    print("EasyOCR ready")
    return reader


def load_trocr(model_name):
    # map our short name to the actual huggingface model path
    hf_paths = {
        'trocr-printed'     : 'microsoft/trocr-base-printed',
        'trocr-handwritten' : 'microsoft/trocr-base-handwritten'
    }
    path = hf_paths[model_name]
    print(f"loading {path}...")

    processor = TrOCRProcessor.from_pretrained(path)
    model = VisionEncoderDecoderModel.from_pretrained(path)
    model.eval()  # inference mode - no gradient updates needed
    print("TrOCR ready")
    return processor, model


def load_model(model_name):
    print(f"loading {model_name} only (skipping others to save memory)")

    if model_name == 'easyocr':
        reader = load_easyocr()
        return {'name': 'easyocr', 'reader': reader}

    elif model_name in ('trocr-printed', 'trocr-handwritten'):
        processor, model = load_trocr(model_name)
        return {'name': model_name, 'processor': processor, 'model': model}


loaded_model = load_model(selected_model)

## Cell 9 - OCR Runner (Agent Step 6)

Run the selected model on the preprocessed image and get back the detected text.

In [ ]:
def run_easyocr(reader, image):
    # readtext runs CRAFT to find text regions then CRNN to read each one
    # lowering the thresholds helps catch low contrast and styled text
    raw = reader.readtext(
        image,
        text_threshold  = 0.4,   # default is 0.7 - lower catches more text regions
        low_text        = 0.3,   # default is 0.4 - lower catches weak character boundaries
        link_threshold  = 0.3,   # default is 0.4 - lower links characters more freely
        contrast_ths    = 0.05,  # default is 0.1 - lower allows low contrast text
        adjust_contrast = 0.7,   # default is 0.5 - higher boosts contrast internally
        detail          = 1,     # return bbox + text + confidence for each detection
        paragraph       = False  # keep each text region separate
    )
    return [(bbox, text, conf) for bbox, text, conf in raw]


def run_trocr(processor, model, image):
    # TrOCR reads one line at a time so we need to find the lines first
    # then crop each line and pass it through the model separately

    def find_text_lines(image):
        # uses OpenCV to find where each line of text is
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

        # make text white and background black, otsu finds the best threshold automatically
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # dilate horizontally so all characters on the same line merge into one blob
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (80, 2))
        dilated = cv2.dilate(binary, kernel, iterations=2)

        contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # filter out tiny blobs that are just noise
        lines = [cv2.boundingRect(c) for c in contours
                 if cv2.boundingRect(c)[2] > 50 and cv2.boundingRect(c)[3] > 10]

        return sorted(lines, key=lambda r: r[1])  # sort top to bottom

    lines = find_text_lines(image)
    h_img, w_img = image.shape[:2]
    results = []

    for x, y, w, h in lines:
        # add a small padding so we dont accidentally cut off letters at the edges
        x1 = max(0, x - 5)
        y1 = max(0, y - 5)
        x2 = min(w_img, x + w + 5)
        y2 = min(h_img, y + h + 5)

        crop = image[y1:y2, x1:x2]
        pil_crop = Image.fromarray(crop)  # TrOCR expects PIL format not numpy

        # processor resizes and normalizes the crop before passing to model
        pixel_values = processor(images=pil_crop, return_tensors='pt').pixel_values

        with torch.no_grad():  # no gradients needed for inference
            generated_ids = model.generate(pixel_values, max_new_tokens=64)

        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        bbox = [[x1,y1],[x2,y1],[x2,y2],[x1,y2]]
        results.append((bbox, text, 1.0))  # TrOCR doesnt return confidence so we use 1.0

    return results


def run_ocr(loaded_model, image):
    name = loaded_model['name']
    print(f"running {name}...")

    if name == 'easyocr':
        results = run_easyocr(loaded_model['reader'], image)
    elif name in ('trocr-printed', 'trocr-handwritten'):
        results = run_trocr(loaded_model['processor'], loaded_model['model'], image)
    else:
        print("unknown model")
        return []

    print(f"found {len(results)} text regions")
    for i, (_, text, conf) in enumerate(results, 1):
        print(f"  {i}. {text}  ({conf:.0%})")

    return results


ocr_results = run_ocr(loaded_model, processed_image)

## Cell 10 - Show Results

In [ ]:
def visualize_results(image, results, model_name):
    # draw a colored box around each detected text region on the image
    fig, ax = plt.subplots(1, figsize=(14, 9))
    ax.imshow(image)

    for bbox, text, conf in results:
        pts = np.array(bbox)
        x_min = pts[:, 0].min()
        y_min = pts[:, 1].min()
        box_w = pts[:, 0].max() - x_min
        box_h = pts[:, 1].max() - y_min

        # green = high confidence, yellow = medium, red = low
        color = 'lime' if conf > 0.7 else ('yellow' if conf > 0.4 else 'red')

        ax.add_patch(patches.Rectangle(
            (x_min, y_min), box_w, box_h,
            linewidth=2, edgecolor=color, facecolor='none'
        ))
        ax.text(x_min, y_min - 5, f"{text} ({conf:.0%})",
                color='white', fontsize=8,
                bbox=dict(facecolor=color, alpha=0.7, pad=1))

    ax.set_title(f"results - {model_name} - {len(results)} regions found")
    ax.axis('off')
    plt.tight_layout()
    plt.show()


def get_final_text(results):
    # join all detected text into one string
    lines = [text for _, text, _ in results]
    raw = '\n'.join(lines)

    # trocr-base-printed outputs everything in uppercase because most of its
    # training data was uppercase - we convert to sentence case to fix this
    fixed = []
    for line in raw.split('\n'):
        line = line.strip()
        if line:
            fixed.append(line[0].upper() + line[1:].lower())
    final_text = '\n'.join(fixed)

    print("=" * 45)
    print("extracted text:")
    print("=" * 45)
    print(final_text)
    print("=" * 45)
    return final_text


visualize_results(processed_image, ocr_results, selected_model)
final_text = get_final_text(ocr_results)

## Cell 11 - Full Agent (all steps in one function)

This wraps everything above into a single function.
Give it any image and it handles the rest automatically.

In [ ]:
def run_agent(image_path):
    # runs the complete pipeline on any image:
    # analyze -> classify -> select model -> preprocess -> load model -> ocr -> output

    print("starting OCR agent...")
    print("-" * 40)

    image      = load_image(image_path)
    features   = analyze_image(image)
    img_type   = classify_image(features)
    model_name = select_model(img_type)
    processed  = preprocess_image(image, model_name)
    model_obj  = load_model(model_name)
    results    = run_ocr(model_obj, processed)
    visualize_results(processed, results, model_name)
    text       = get_final_text(results)

    print("-" * 40)
    print(f"image type : {img_type}")
    print(f"model used : {model_name}")
    print(f"words found: {len(text.split())}")
    return text


extracted_text = run_agent(IMAGE_PATH)

## Cell 12 - Gradio Web App

Wraps the agent in a simple web UI so anyone can use it without touching the code.
Running this cell gives you a public link you can share.

In [ ]:
def agent_for_gradio(uploaded_image):
    # gradio passes the uploaded image as a numpy array directly
    # so we skip load_image() and go straight to analyze
    features   = analyze_image(uploaded_image)
    img_type   = classify_image(features)
    model_name = select_model(img_type)
    processed  = preprocess_image(uploaded_image, model_name)
    model_obj  = load_model(model_name)
    results    = run_ocr(model_obj, processed)
    text       = get_final_text(results)

    # save output to a text file so user can download it
    output_file = "extracted_text.txt"
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(text)

    return img_type, model_name, text, output_file


demo = gr.Interface(
    fn      = agent_for_gradio,
    inputs  = gr.Image(type="numpy", label="Upload your image"),
    outputs = [
        gr.Textbox(label="Image Type"),
        gr.Textbox(label="Model Used"),
        gr.Textbox(
            label="Extracted Text",
            lines=10,
            max_lines=50,
            show_copy_button=True
        ),
        gr.File(label="Download as .txt")
    ],
    title       = "OCR AI Agent",
    description = "Upload any image with text in it. The agent figures out what kind of image it is, picks the right OCR model, and extracts the text.",
    examples    = [[IMAGE_PATH]],
    allow_flagging = "never"
)

demo.launch(share=True)  # share=True gives a public URL valid for 72 hours